# EDA On `player_play.csv` Dataset


In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    RAW_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

In [ ]:
# ==========================================
# Load Data
# ==========================================

pp_df = (
    pl.read_csv(
        str(RAW_DATA_DIR / "player_play.csv"), 
        # Change "NA" to None
        null_values = {
            "home_team_win_probability_added": "NA", 
            "visitor_team_win_probility_added": "NA",
            "pre_snap_home_team_win_probability": "NA", 
            "pre_snap_visitor_team_win_probability": "NA"
        }
    )
)

# ===========================================
# Clean Data
# ===========================================

# --------------------------------------
# Convert all "NA" values to missing
# --------------------------------------
na_cols = [col for col in pp_df.columns if pp_df[col].cast(pl.String).eq("NA").any()]

pp_df = (
    pp_df
    .with_columns([
        pl.when(pl.col(col).cast(pl.String) == "NA")
        .then(None)
        .otherwise(pl.col(col))
        .alias(col)
        for col in na_cols
    ])
)

# --------------------------------------
# Fix a misspelled variable name
# --------------------------------------
pp_df = (
    pp_df
    .rename({"visitor_team_win_probility_added": "visitor_team_win_probability_added"})
)

# --------------------------------------
# Convert columns to proper schema
# --------------------------------------

pp_df = (
    pp_df
    # 1. game_id
    #    Looks good
    # 2. play_id
    #    Looks good
    # 3. team_abbr
    #    Looks good
    # 4. nfl_id
    #    Looks good
    # 5. lined_up_position
    #    Looks good
    # 6. play_description
    #    Looks good
    # 7. quarter
    #    Looks good
    # 8. down
    #    Looks good
    # 9. yards_to_g
    #    Looks good
    # 10. possession_team
    #     Looks good
    # 11. yardline_side
    #     Looks good
    # 12. yardline_number
    #     Looks good
    # 13. game_clock
    #     Looks good
    # 14. pre_snap_home_score
    #     Looks good
    # 15. pre_snap_visitor_score
    #     Looks good
    # 16. play_direction
    #     Looks good
    # 17. pre_snap_home_team_win_probability
    #     Looks good
    # 18. pre_snap_visitor_team_win_probability
    #     Looks good
    # 19. home_team_win_probability_added
    #     Looks good
    # 20. visitor_team_win_probability_added
    #     Looks good
    # 21. expected_points_added
    #     Looks good
    # 22. expected_points
    #     Looks good
    # 23. offense_formation
    #     Looks good
    # 24. receiver_alignment
    #     Looks good
    # 25. pass_result
    #     Looks good
    # 26. pass_length
    #     Convert from string to float64
    .with_columns(pl.col("pass_length").cast(pl.Float64))
    # 27. play_nullified_by_penalty
    #     Convert to boolean instead of "Y"/"N"
    .with_columns(pl.col("play_nullified_by_penalty") == "Y")
    # 28. penalty_yards
    #     Convert from string to float64
    .with_columns(pl.col("penalty_yards").cast(pl.Float64))
    # 29. pre_penalty_yards_gained
    #     Looks good
    # 30. yards_gained
    #     Looks good
    # 31. team_coverage_man_zone
    #     Looks good
    # 32. team_coverage_type
    #     Looks good
    # 33. in_motion_at_ball_snap
    #     Convert from string to boolean
    .with_columns(pl.col("in_motion_at_ball_snap").replace_strict({"TRUE": True, "FALSE": False}, return_dtype=pl.Boolean))
    # 34. target
    #     Looks good
    # 35. rec_yards
    #     Convert from string to Float64
    .with_columns(pl.col("rec_yards").cast(pl.Float64))
    # 36. rush_yards
    #     Convert from string to Float64
    .with_columns(pl.col("rush_yards").cast(pl.Float64))
    # 37. yards_after_catch
    #     Convert from string to Float64
    .with_columns(pl.col("yards_after_catch").cast(pl.Float64))
    # 38. fumble
    #     Convert fumble from String (1/null) to Boolean (True/null)
    .with_columns(fumble = pl.when(pl.col("fumble") == "1").then(True).otherwise(None).cast(pl.Boolean))
    # 39. fumble_lost
    #     Convert fumble_lost from String (1/0/null) to Boolean(True/False/null)
    .with_columns(fumble_lost = pl.when(pl.col("fumble_lost") == "1").then(True).when(pl.col("fumble_lost") == "0").then(False).otherwise(None).cast(pl.Boolean))
    # 40. route_ran
    #     Looks good
    # 41. separation_at_pass_forward
    #     Convert from String to Float64
    .with_columns(pl.col("separation_at_pass_forward").cast(pl.Float64))
    # 42. cushion
    #     Convert from String to Float64
    .with_columns(pl.col("cushion").cast(pl.Float64))
    # 43. expected_yards_after_catch
    #     Convert from String to Float64
    .with_columns(pl.col("expected_yards_after_catch").cast(pl.Float64))
    # 44. dropback_duration
    #     Convert from String to Float64
    .with_columns(pl.col("dropback_duration").cast(pl.Float64))
    # 45. extended_sack_allowed
    #     Convert from String to Float64 (0, 0.5, 1, null)
    .with_columns(pl.col("extended_sack_allowed").cast(pl.Float64))
    # 46. pass_rushers_encountered
    #     Convert from String to Float64
    .with_columns(pl.col("pass_rushers_encountered").cast(pl.Float64))
    # 47. peak_pressure_probaility_allowed
    #     Convert from String to Float64
    .with_columns(pl.col("pass_rushers_encountered").cast(pl.Float64))
    # 48. pressure_allowed
    #     Convert from String to Boolean
    .with_columns(pl.col("pressure_allowed").replace_strict({"TRUE": True, "FALSE": False}, return_dtype=pl.Boolean))
    # 49. sack_allowed
    #     Convert from String to Float64
    .with_columns(pl.col("sack_allowed").cast(pl.Float64))
    # 50. time_to_pressure_allowed
    #     Convert from String to Float64
    .with_columns(pl.col("time_to_pressure_allowed").cast(pl.Float64))
    # 51. quick_pressure
    #     Convert from String to Boolean
    .with_columns(pl.col("quick_pressure").replace_strict({"TRUE": True, "FALSE": False}, return_dtype=pl.Boolean))
    # 52. unblocked_pressure
    #     Convert from String to Boolean
    .with_columns(pl.col("unblocked_pressure").replace_strict({"TRUE": True, "FALSE": False}, return_dtype=pl.Boolean))
    # 53. blitzing
    #     Convert from String to Boolean
    .with_columns(pl.col("blitzing").replace_strict({"TRUE": True, "FALSE": False}, return_dtype=pl.Boolean))
    # 54. player_get_off
    #     Convert from String to Float64
    .with_columns(pl.col("player_get_off").cast(pl.Float64))
    # 55. sack
    #     Convert from String to Float (0, 0.5, 1, null)
    .with_columns(pl.col("sack").cast(pl.Float64))
    # 56. tackle
    #     Change value of 2 to 1, there shouldn't be a value of 2
    .with_columns(tackle = pl.when(pl.col("tackle") == "2").then(pl.lit("1")).otherwise(pl.col("tackle")))
    #     Convert from String (1/0/null) to Boolean (True/False/null)
    .with_columns(tackle = pl.when(pl.col("tackle") == "1").then(True).when(pl.col("tackle") == "0").then(False).otherwise(None).cast(pl.Boolean))
    # 57. assist
    #     Convert from String to Boolean
    .with_columns(assist = pl.when(pl.col("assist") == "1").then(True).when(pl.col("assist") == "0").then(False).otherwise(None).cast(pl.Boolean))
    # 58. caused_forced_fumble
    #     Convert caused_forced_fumble from String (1/0/null) to Boolean(True/False/null)
    .with_columns(caused_forced_fumble = pl.when(pl.col("caused_forced_fumble") == "1").then(True).when(pl.col("caused_forced_fumble") == "0").then(False).otherwise(None).cast(pl.Boolean))
    # 59. recovered_fumble
    #     Convert from String to Boolean
    .with_columns(pl.col("recovered_fumble").replace_strict({"TRUE": True, "FALSE": False}, return_dtype=pl.Boolean))
    # 60. tackle_for_loss
    #     Convert from String to Boolean
    .with_columns(pl.col("tackle_for_loss").replace_strict({"TRUE": True, "FALSE": False}, return_dtype=pl.Boolean))
    # 61. time_to_pressure
    #     Convert from String to Float64
    .with_columns(pl.col("time_to_pressure").cast(pl.Float64))
    # 62. time_to_qb_hurry
    #     Convert from String to Float64
    .with_columns(pl.col("time_to_qb_hurry").cast(pl.Float64))
    # 63. coverage_assignment
    #     Looks good
    # 64. converage_assignment_at_snap
    #     Looks good
)


print("Player-Plays Data:")
print(f"    - Dimensions: {pp_df.shape}")
print(f"    - Number of Unique Player IDs: {pp_df.select("nfl_id").n_unique()}")
pp_df

In [ ]:
# =======================================
# Missing Data
# =======================================

# Many variables are expected to have missing data
# For example, a WR will have missing data on the sack variable
# because sacks don't apply to an offensive player

pp_df.select(pl.all().is_null().sum())

In [ ]:
def count_missing(var: str, df: pl.DataFrame = pp_df) -> None:
    print(f"Number of missing values: {df.select(pl.col(var).is_null().sum()).item()}")

def min_max(var: str):
    print(f"Min & Max Values: {pp_df.select(var).min().item()}, {pp_df.select(var).max().item()}")

In [ ]:
# Add game data to the player-plays data for EDA

games_df = pl.read_csv(str(RAW_DATA_DIR / "games.csv"))

pp_df_with_games = (
    pp_df.join(games_df, on = "game_id", how = "left")
)

<br><br/>
## Play Context & Identifiers
***

In [ ]:
# ==============================
# game_id
# ==============================

count_missing("game_id")

In [ ]:
# ==============================
# play_id
# ==============================

count_missing("play_id")

In [ ]:
# ==============================
# team_abbr
# ==============================

# NOTE NOTE NOTE: Sometimes the Rams (LAR) and Chargers (LAC) are put down as "LA"

count_missing("team_abbr")
print("")
print(f"Number of Teams: {pp_df.select("team_abbr").n_unique()}")
print(sorted(pp_df.select("team_abbr").unique().to_series().to_list()))

In [ ]:
# ==============================
# nfl_id
# ==============================

# NOTE NOTE NOTE: Some players are missing from this table. But we also know that
#                 some players get no exposure.

count_missing("nfl_id")
print("")

print(f"Number of Unique Players: {pp_df.select("nfl_id").n_unique()}")



In [ ]:
# ==============================
# lined_up_position
# ==============================

count_missing("lined_up_position")
print("")

print(f"Number of Unique Values: {pp_df.select("lined_up_position").n_unique()}")
print(sorted(pp_df.filter(pl.col("lined_up_position").is_not_null()).select("lined_up_position").unique().to_series().to_list()))

In [ ]:
# ==============================
# play_description
# ==============================

count_missing("play_description")

In [ ]:
# ==============================
# quarter
# ==============================

count_missing("quarter")
print("")
print(f"Values: {sorted(pp_df.select("quarter").unique().to_series().to_list())}")

In [ ]:
# ==============================
# down
# ==============================

count_missing("down")
print("")
print(f"Values: {sorted(pp_df.select("down").unique().to_series().to_list())}")
pp_df.group_by("down").agg(count = pl.len()).sort("down")

In [ ]:
# ==============================
# yards_to_go
# ==============================

count_missing("yards_to_go")
print("")
min_max("yards_to_go")

In [ ]:
# ==============================
# possession_team
# ==============================

# NOTE NOTE NOTE: Sometimes the Rams (LAR) and Chargers (LAC) are put down as "LA"

count_missing("possession_team")
print("")
print(f"Number of Teams: {pp_df.select("possession_team").n_unique()}")
print(sorted(pp_df.select("possession_team").unique().to_series().to_list()))

In [ ]:
# ==============================
# yardline_side
# ==============================

# NOTE NOTE NOTE: Sometimes the Rams (LAR) and Chargers (LAC) are put down as "LA"

count_missing("yardline_side")
print("")
print(f"Number of Teams (not missing): {pp_df.filter(pl.col("yardline_side").is_not_null()).select("yardline_side").n_unique()}")
print(sorted(pp_df.filter(pl.col("yardline_side").is_not_null()).select("yardline_side").unique().to_series().to_list()))

In [ ]:
# ==============================
# yardline_number
# ==============================

count_missing("yardline_number")
print("")
min_max("yardline_number")

In [ ]:
# ==============================
# game_clock
# ==============================

count_missing("game_clock")

In [ ]:
# ==============================
# pre_snap_home_score
# ==============================

count_missing("pre_snap_home_score")
print("")
min_max("pre_snap_home_score")

In [ ]:
# ==============================
# pre_snap_visitor_score
# ==============================

count_missing("pre_snap_visitor_score")
print("")
min_max("pre_snap_visitor_score")

In [ ]:
# ==============================
# play_direction
# ==============================

count_missing("play_direction")
print("")
pp_df.group_by("play_direction").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count"))

In [ ]:
# ==============================
# pre_snap_home_team_win_probability
# ==============================

count_missing("pre_snap_home_team_win_probability")
print("")
min_max("pre_snap_home_team_win_probability")

In [ ]:
# ==============================
# pre_snap_visitor_team_win_probability
# ==============================

count_missing("pre_snap_visitor_team_win_probability")
print("")
min_max("pre_snap_visitor_team_win_probability")

In [ ]:
# ==============================
# home_team_win_probability_added
# ==============================

# NOTE NOTE NOTE: The value of 0.99308 seems wrong. The play description doesn't describe
#                 an influential play. Not sure how it got this extreme estimate.

count_missing("home_team_win_probability_added")
print("")
min_max("home_team_win_probability_added")

# fig = px.violin(
#     pp_df, 
#     y = "home_team_win_probability_added", 
#     points = "all",
# )

# fig.show()

In [ ]:
# ==============================
# visitor_team_win_probability_added
# ==============================

# NOTE NOTE NOTE: Same issue as home_team_win_probability_added

count_missing("visitor_team_win_probability_added")
print("")
min_max("visitor_team_win_probability_added")

In [ ]:
# ==============================
# expected_points_added
# ==============================

# From what I saw, the extreme EPA values are legit. For example a 
# EPA = -12.89 came from a 99 yard pick six

count_missing("expected_points_added")
print("")
min_max("expected_points_added")

In [ ]:
# ==============================
# expected_points
# ==============================

count_missing("expected_points")
print("")
min_max("expected_points")

In [ ]:
# ==============================
# offense_formation
# ==============================

count_missing("offense_formation")
print("")

print(f"Number of Unique Values: {pp_df.select("offense_formation").n_unique()}")
print(sorted(pp_df.filter(pl.col("offense_formation").is_not_null()).select("offense_formation").unique().to_series().to_list()))

In [ ]:
# ==============================
# receiver_alignment
# ==============================

count_missing("receiver_alignment")
print("")

print(f"Number of Unique Values: {pp_df.select("receiver_alignment").n_unique()}")
print(sorted(pp_df.filter(pl.col("receiver_alignment").is_not_null()).select("receiver_alignment").unique().to_series().to_list()))

In [ ]:
# ==============================
# pass_result
# ==============================

count_missing("pass_result")
print("")

pp_df.group_by("pass_result").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# pass_length
# ==============================

# NOTE NOTE NOTE: pass_length has some incorrect data. 
#                 For example the play where pass_length = -70 was just an incomplete pass

count_missing("pass_length")
print("")
min_max("pass_length")

In [ ]:
# ==============================
# play_nullified_by_penalty
# ==============================

count_missing("play_nullified_by_penalty")

pp_df.group_by("play_nullified_by_penalty").agg(pl.len())

In [ ]:
# ==============================
# penalty_yards
# ==============================

# NOTE NOTE NOTE: penalty_yards looks good except I don't know how they determine if the value
#                 is positive or negative. It must be the direction the play is moving but not sure

count_missing("penalty_yards")
print("")
min_max("penalty_yards")

In [ ]:
# ==============================
# pre_penalty_yards_gained
# ==============================

# NOTE NOTE NOTE: Sometimes there's data for pre_penalty_yards
#                 gained even if a penalty didn't happen

count_missing("pre_penalty_yards_gained")
print("")
min_max("pre_penalty_yards_gained")

In [ ]:
# ==============================
# yards_gained
# ==============================

count_missing("yards_gained")
print("")
min_max("yards_gained")

In [ ]:
# ==============================
# team_coverage_man_zone
# ==============================

count_missing("team_coverage_man_zone")
print("")

pp_df.group_by("team_coverage_man_zone").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# team_coverage_type
# ==============================

count_missing("team_coverage_type")
print("")

pp_df.group_by("team_coverage_type").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# in_motion_at_ball_snap
# ==============================

count_missing("in_motion_at_ball_snap")
print("")

pp_df.group_by("in_motion_at_ball_snap").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# target
# ==============================

count_missing("target")
print("")

pp_df.group_by("target").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# rec_yards
# ==============================

count_missing("rec_yards")
print("")
min_max("rec_yards")

In [ ]:
# ==============================
# rush_yards
# ==============================

count_missing("rush_yards")
print("")
min_max("rush_yards")

In [ ]:
# ==============================
# yards_after_catch
# ==============================

count_missing("yards_after_catch")
print("")
min_max("yards_after_catch")

In [ ]:
# ==============================
# fumble
# ==============================

count_missing("fumble")
print("")

pp_df.group_by("fumble").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# fumble_lost
# ==============================

# NOTE NOTE NOTE: some fumbles don't have data for fumble_lost

count_missing("fumble_lost")
print("")

pp_df.group_by("fumble_lost").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# route_ran
# ==============================

count_missing("route_ran")
print("")
print(f"Number of Routes (non-missing): {pp_df.filter(pl.col("route_ran").is_not_null()).select(pl.col("route_ran")).n_unique()}")
print(sorted(pp_df.filter(pl.col("route_ran").is_not_null()).select(pl.col("route_ran")).unique().to_series().to_list()))

pp_df.group_by("route_ran").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# separation_at_pass_forward
# ==============================

# NOTE: Some separation_at_pass_forward values aren't quality measurements.
#       For example, one a hail mary play where all defenders are waiting in the end
#       zone, if the offense makes a short pass and tries to lateral, then the 
#       separation at pass will be huge, even though the receiver didn't do anything.
#       Should take cushion into account

count_missing("separation_at_pass_forward")
print("")
min_max("separation_at_pass_forward")

In [ ]:
# ==============================
# cushion
# ==============================

count_missing("cushion")
print("")
min_max("cushion")

In [ ]:
# ==============================
# expected_yards_after_catch
# ==============================

count_missing("expected_yards_after_catch")
print("")
min_max("expected_yards_after_catch")

In [ ]:
# ==============================
# dropback_duration
# ==============================

count_missing("dropback_duration")
print("")
min_max("dropback_duration")

In [ ]:
# ==============================
# extended_sack_allowed
# ==============================

count_missing("extended_sack_allowed")
print("")

pp_df.group_by("extended_sack_allowed").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# pass_rushers_encountered
# ==============================

count_missing("pass_rushers_encountered")
print("")

pp_df.group_by("pass_rushers_encountered").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# peak_pressure_probability_allowed
# ==============================

count_missing("peak_pressure_probability_allowed")
print("")
min_max("peak_pressure_probability_allowed")

In [ ]:
# ==============================
# pressure_allowed
# ==============================

count_missing("pressure_allowed")
print("")

pp_df.group_by("pressure_allowed").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# sack_allowed
# ==============================

count_missing("sack_allowed")
print("")

pp_df.group_by("sack_allowed").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# time_to_pressure_allowed
# ==============================

count_missing("time_to_pressure_allowed")
print("")
min_max("time_to_pressure_allowed")

In [ ]:
# ==============================
# quick_pressure
# ==============================

count_missing("quick_pressure")
print("")

pp_df.group_by("quick_pressure").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# unblocked_pressure
# ==============================

count_missing("unblocked_pressure")
print("")

pp_df.group_by("unblocked_pressure").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# blitzing
# ==============================

count_missing("blitzing")
print("")

pp_df.group_by("blitzing").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# player_get_off
# ==============================

count_missing("player_get_off")
print("")
min_max("player_get_off")

In [ ]:
# ==============================
# sack
# ==============================

count_missing("sack")
print("")

pp_df.group_by("sack").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# tackle
# ==============================

count_missing("tackle")
print("")

pp_df.group_by("tackle").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# assist
# ==============================

count_missing("assist")
print("")

pp_df.group_by("assist").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# caused_forced_fumble
# ==============================

count_missing("caused_forced_fumble")
print("")

pp_df.group_by("caused_forced_fumble").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# tackle_for_loss
# ==============================

count_missing("tackle_for_loss")
print("")

pp_df.group_by("tackle_for_loss").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# time_to_pressure
# ==============================

count_missing("time_to_pressure")
print("")
min_max("time_to_pressure")

In [ ]:
# ==============================
# time_to_qb_hurry
# ==============================

count_missing("time_to_qb_hurry")
print("")
min_max("time_to_qb_hurry")

In [ ]:
# ==============================
# coverage_assignment
# ==============================

count_missing("coverage_assignment")
print("")
print(f"Number of Routes (non-missing): {pp_df.filter(pl.col("coverage_assignment").is_not_null()).select(pl.col("coverage_assignment")).n_unique()}")
print(sorted(pp_df.filter(pl.col("coverage_assignment").is_not_null()).select(pl.col("coverage_assignment")).unique().to_series().to_list()))

pp_df.group_by("coverage_assignment").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)

In [ ]:
# ==============================
# coverage_assignment_at_snap
# ==============================

count_missing("coverage_assignment_at_snap")
print("")
print(f"Number of Routes (non-missing): {pp_df.filter(pl.col("coverage_assignment_at_snap").is_not_null()).select(pl.col("coverage_assignment_at_snap")).n_unique()}")
print(sorted(pp_df.filter(pl.col("coverage_assignment_at_snap").is_not_null()).select(pl.col("coverage_assignment_at_snap")).unique().to_series().to_list()))

pp_df.group_by("coverage_assignment_at_snap").agg(count = pl.len()).with_columns(perc = 100 * pl.col("count") / pl.sum("count")).sort("perc", descending=True)